# Run Forge training on Modal
This notebook defines a Modal app and function that run the Forge pretraining and fine-tuning pipeline remotely.

To run on Modal, export this notebook as a Python script (e.g. `forge_modal.py`) and then run:
- `modal run forge_modal.py::main`

# Imports

In [ ]:
import modal
from modal import App, Image, Secret

from forge.embeddings import Forge
from forge.pipeline import pretrain, finetune_integral_gap
from forge.utils import Constants

import os

# Data

In [ ]:
# Name of the Modal Secret that holds your HF token
HF_SECRET = "hf-token"  # TODO: create this secret in Modal and store HF_TOKEN in it

# Modal app and base image definition
app = App("forge-modal-training")

image = (
    Image.debian_slim(python_version="3.12")
    .pip_install("torch", "pyyaml", "tqdm", "numpy", "pandas", "scikit-learn", "scipy", "huggingface-hub", "gurobi", "torch-geometric", "vector-quantize-pytorch")
)

# Pre-Train

In [ ]:
@app.function(image=image, secrets=[Secret.from_name(HF_SECRET_NAME)], timeout=60 * 60 * 10)
def run_pretrain_and_finetune():
    """Download data, pretrain Forge, then fine-tune on integral gap using Modal."""
    # Get token from environment (provided via Modal Secret)
    token = os.environ.get("HF_TOKEN", HF_SECRET)

    # Clone the Forge GitHub repository when running in a fresh environment
    # If the directory already exists, this is a no-op.
    import os, subprocess

    REPO_URL = "https://github.com/skadio/forge.git"
    REPO_DIR = "forge"

    if not os.path.exists(REPO_DIR):
        subprocess.check_call(["git", "clone", REPO_URL])
        os.chdir(REPO_DIR)
    else:
        os.chdir(REPO_DIR)

    # Download MIP instances
    # NOTE: path is relative to the project root when running on Modal
    subprocess.call(f"python data/download.py --token {token}", shell=True)

    # Forge model with its pre-training configuration
    forge = Forge(train_config_yaml="./forge/configs/train_config.yaml")

    # Pretrain Forge on a set of MIP instances in the given input folder
    # The pretrained model pickle is stored as output_forge_pretrained_pkl
    # The intermediate mip_to_mipinfo pickle is stored as output_mip_to_mipinfo_pkl
    pretrain(
        forge=forge,
        input_mip_folder="./data/instances/",
        input_mip_instances_file="./data/configs/pretrain.txt",
        output_mip_to_mipinfo_pkl="./models/iclr26_pretrain_mip_to_mipinfo.pkl",
        output_forge_pretrained_pkl="./models/forge_pretrained.pkl",
        output_log_file="./models/forge_pretrained.log",
    )

    # Reload Forge with the same training config for fine-tuning
    forge = Forge(train_config_yaml="./forge/configs/train_config.yaml")

    # Fine-tune Forge to predict integral gaps
    finetune_integral_gap(
        forge=forge,
        input_forge_pkl="./models/forge_pretrained.pkl",
        model_type=Constants.FORGE_FINE_TUNE_INTEGRAL_GAP,
        input_mip_folder="./data/instances/",
        input_mip_instances_file="./data/configs/fine_tune_integral_gap.txt",
        output_forge_finetuned_pkl="./models/forge_integral_gap.pkl",
        output_mip_to_gapinfo_pkl="./models/mip_to_gapinfo.pkl",
        num_parallel_workers=10
    )

# Fine Tune

In [ ]:
@app.local_entrypoint()
def main():
    """Entry point used when running via `modal run`.","""
    run_pretrain_and_finetune.remote()

# Notebook Run 

In [ ]:
with modal.enable_output():
    with app.run():
        run_pretrain_and_finetune.remote()